# Pre-Goal-1 toy: black-box memory eval on 2 LongMemEval questions

**What this shows:** memdiag scoring an agent's memory from `(query, context, response)` plus the user's
logged history, **without the gold answer**, and then checking its verdicts against the gold labels.

Everything runs locally with an open-source model through Ollama. This is a 2-question smoke test,
not a benchmark result.

| Piece | What it is here |
|---|---|
| Dataset | LongMemEval-S (cleaned, MIT license), 2 questions, ~40 history sessions each |
| Memory systems under test | `bm25_top3` (keyword search over past user messages) and `recent_2_sessions` (only the last 2 sessions) |
| Model | Qwen2.5 via Ollama (`qwen2.5:7b` by default; `qwen2.5:3b` also works) |

### What Qwen is used for (4 separate roles)

| Role | Side | Input | Output |
|---|---|---|---|
| `answerer` | the **agent under test** (stands in for a production agent) | memory context + question | the response |
| `extractor` | **memdiag** (the open-source LLM extractor) | the user's messages, one session at a time | short dated facts -> the fact ledger |
| `judge_sufficiency` | **memdiag** | ledger facts + context + question | yes/no: did the context hold what the question needs? |
| `judge_use` | **memdiag** | context + question + response | yes/no: did the response use the context correctly? |
| `gold_judge` | **gold labels** (never shown to memdiag) | question + gold answer + response | yes/no: is the response correct? (LongMemEval-style prompt) |

Caveat: one model plays every role, so the judge may be lenient toward its own answers. Goal 1 needs a
separate judge model and a hand-labeled audit; here, read the printed outputs and check them yourself.

## 1. Setup
Requires `ollama serve` running and the model pulled (`ollama pull qwen2.5:7b`).

In [1]:
import json
import os
import re
import sys
import time
import urllib.request
from collections import defaultdict
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
sys.path.insert(0, str(REPO))
from memdiag.detect import BM25  # noqa: E402  (memdiag's dependency-free BM25)

MODEL = os.environ.get("MEMDIAG_MODEL", "qwen2.5:7b")
OLLAMA = "http://localhost:11434/v1/chat/completions"   # OpenAI-compatible endpoint
DATA = REPO / "experiments" / "data"
DATA.mkdir(parents=True, exist_ok=True)
QIDS = ["852ce960", "8550ddae"]   # one knowledge-update, one single-session-user question
print("model:", MODEL)

model: qwen2.5:7b


## 2. Data
Downloads LongMemEval-S cleaned (277 MB) once into `experiments/data/` (git-ignored).

In [2]:
src = DATA / "longmemeval_s_cleaned.json"
if not src.exists():
    url = "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json"
    urllib.request.urlretrieve(url, src)
items = {x["question_id"]: x for x in json.load(open(src)) if x["question_id"] in QIDS}
for q in QIDS:
    x = items[q]
    print(f"{q} [{x['question_type']}] {x['question']}")
    print(f"   gold answer: {x['answer']} | {len(x['haystack_sessions'])} sessions | evidence: {x['answer_session_ids']}")

852ce960 [knowledge-update] What was the amount I was pre-approved for when I got my mortgage from Wells Fargo?
   gold answer: $400,000 | 39 sessions | evidence: ['answer_3a6f1e82_1', 'answer_3a6f1e82_2']
8550ddae [single-session-user] What type of cocktail recipe did I try last weekend?
   gold answer: lavender gin fizz | 42 sessions | evidence: ['answer_c8354ae9']


## 3. Every Qwen call goes through one function, tagged with its role

In [3]:
LOG = []   # one entry per LLM call: role, seconds, prompt and output (truncated)

def chat(role, prompt, as_json=False):
    body = {"model": MODEL, "temperature": 0, "seed": 0, "messages": [{"role": "user", "content": prompt}]}
    if as_json:
        body["response_format"] = {"type": "json_object"}
    req = urllib.request.Request(OLLAMA, json.dumps(body).encode(), {"Content-Type": "application/json"})
    t = time.time()
    with urllib.request.urlopen(req, timeout=600) as r:
        out = json.load(r)["choices"][0]["message"]["content"]
    LOG.append({"role": role, "secs": round(time.time() - t, 1), "prompt": prompt[-400:], "output": out[:400]})
    return out

def yes(text):
    return text.strip().lower().startswith("yes")

def as_obj(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group(0)) if m else {}

print(chat("smoke", "Reply with the single word: ready"))

Ready


## 4. The agent under test: two memory systems + Qwen as the answerer
This part plays the production agent. memdiag only sees what it logs: query, context, response.

In [4]:
def user_chunks(item):
    for sid, date, sess in zip(item["haystack_session_ids"], item["haystack_dates"], item["haystack_sessions"], strict=True):
        for turn in sess:
            if turn["role"] == "user":
                yield {"session": sid, "date": date, "text": turn["content"][:1200]}

def mem_bm25(item, k=3):
    chunks = list(user_chunks(item))
    sc = BM25([c["text"] for c in chunks]).scores(item["question"])
    return [chunks[i] for i in sorted(range(len(chunks)), key=lambda i: -sc[i])[:k]]

def mem_recent(item, n_sessions=2):
    keep = set(item["haystack_session_ids"][-n_sessions:])
    return [c for c in user_chunks(item) if c["session"] in keep][-6:]

CONFIGS = {"bm25_top3": mem_bm25, "recent_2_sessions": mem_recent}

def fmt(ctx):
    return "\n".join(f"- [{c['date']}] {c['text'][:600]}" for c in ctx)

def answer(item, ctx):
    return chat("answerer",
                f"You are a helpful assistant with long-term memory.\nMemories about the user:\n{fmt(ctx)}\n\n"
                f"Current date: {item['question_date']}\nUser: {item['question']}\nAnswer in one or two sentences.")

calls = []   # the production log: exactly what memdiag gets
for q in QIDS:
    for name, mem in CONFIGS.items():
        ctx = mem(items[q])
        calls.append({"qid": q, "config": name, "query": items[q]["question"], "context": ctx,
                      "response": answer(items[q], ctx)})
for c in calls:
    print(f"[{c['qid']} / {c['config']}] {c['response']}")

[852ce960 / bm25_top3] You were pre-approved for $400,000 from Wells Fargo during your previous interaction, but you mentioned getting pre-approved for $350,000 this time for the house you're buying.
[852ce960 / recent_2_sessions] I don't have access to specific financial information about individual users, including pre-approval amounts for mortgages from Wells Fargo. You may want to check your personal records or contact Wells Fargo directly for that information.
[8550ddae / bm25_top3] Last weekend, you tried making a lavender gin fizz but had to improvise with honey and water since you didn't have simple syrup on hand.
[8550ddae / recent_2_sessions] I don't have specific information about the cocktails you tried last weekend, as that detail wasn't part of our previous conversation. If you'd like, you can share more details about it, and I'll do my best to assist you!


## 5. memdiag, step 1: the open-source LLM extractor builds a fact ledger from the user's history
This is the reference memdiag checks against. It is built offline from logs, once per session, and is
not the agent's memory: it has no token budget and sees every session. Cached to disk, since it is the
slow step (~40 calls per question).

In [5]:
def extract_ledger(item):
    ledger = []
    for sid, date, sess in zip(item["haystack_session_ids"], item["haystack_dates"], item["haystack_sessions"], strict=True):
        said = "\n".join(t["content"][:800] for t in sess if t["role"] == "user")
        out = as_obj(chat("extractor",
            "Extract the personal facts the USER states about themselves (events, numbers, possessions, "
            'preferences, plans). Ignore general questions. Return JSON {"facts": ["short fact", ...]} '
            "(empty list if none).\n\nUser messages:\n" + said, as_json=True))
        for f in out.get("facts", []) or []:
            if isinstance(f, str) and f.strip():
                ledger.append({"session": sid, "date": date, "fact": f.strip()})
    return ledger

ledgers = {}
for q in QIDS:
    cache = DATA / f"ledger_{q}_{MODEL.replace(':', '_')}.json"
    if cache.exists():
        ledgers[q] = json.load(open(cache))
    else:
        t = time.time()
        ledgers[q] = extract_ledger(items[q])
        json.dump(ledgers[q], open(cache, "w"), indent=1)
        print(f"{q}: extracted in {time.time() - t:.0f}s")
    print(f"{q}: {len(ledgers[q])} facts, e.g.", [f["fact"] for f in ledgers[q][:3]])

852ce960: 137 facts, e.g. ['trying to establish a consistent bedtime routine', 'struggling with waking up at the desired time', 'reading a book for at least 20 minutes before bed helps relax and reduce screen time']
8550ddae: 202 facts, e.g. ['visited the Modern Art Museum on Saturday, March 12th', 'enjoyed a Pollock piece for 20 minutes', 'interested in learning more about abstract expressionism']


## 6. memdiag, step 2: look up the relevant facts, then ask the judge two narrow questions
Verdict: **pass** if sufficient and used. Otherwise the failure is located at
**retrieve** (context lacked the needed fact) or **use** (it was there, but the response didn't use it).

In [6]:
def reference(ledger, question, k=6):
    sc = BM25([f["fact"] for f in ledger]).scores(question)
    top = [ledger[i] for i in sorted(range(len(ledger)), key=lambda i: -sc[i])[:k] if sc[i] > 0]
    return sorted(top, key=lambda f: f["date"])

def memdiag_judge(call, ref):
    ref_txt = "\n".join(f"- [{f['date']}] {f['fact']}" for f in ref) or "- (nothing found)"
    suff = chat("judge_sufficiency",
        "You audit an assistant's memory. Below are facts the user stated in past conversations (dated; later "
        "facts override earlier ones), the memory context the assistant was given, and the user's question."
        f"\n\nUser's past facts:\n{ref_txt}\n\nContext given to assistant:\n{fmt(call['context'])}\n\n"
        f"Question: {call['query']}\n\nDoes the context contain the information needed to answer the "
        "question, consistent with the user's LATEST stated facts? Answer yes or no, then quote the supporting "
        "line from the context.")
    used = chat("judge_use",
        f"Context:\n{fmt(call['context'])}\n\nQuestion: {call['query']}\nResponse: {call['response']}\n\n"
        "Does the response answer the question using the information in the context, without contradicting "
        "it? Answer yes or no, then one sentence why.")
    s, u = yes(suff), yes(used)
    return {"sufficient": s, "used": u, "pass": s and u,
            "location": None if s and u else ("retrieve" if not s else "use"),
            "why_sufficiency": suff[:250], "why_use": used[:250]}

for c in calls:
    c["reference"] = reference(ledgers[c["qid"]], c["query"])
    c["memdiag"] = memdiag_judge(c, c["reference"])
    m = c["memdiag"]
    print(f"[{c['qid']} / {c['config']}] pass={m['pass']} location={m['location']}")
    print("   reference facts:", [f["fact"] for f in c["reference"]])
    print("   sufficiency:", m["why_sufficiency"].replace("\n", " "))
    print("   use:        ", m["why_use"].replace("\n", " "))

[852ce960 / bm25_top3] pass=False location=use
   reference facts: ['pre-approved for $350,000 from Wells Fargo', 'user was recently promoted to senior software engineer', 'aware that the series has historical inspirations from Renaissance Italy', 'previous 5-day trip to Paris last month was amazing', 'recently got car serviced on the 15th of last month', 'pre-approved for $400,000 from Wells Fargo']
   sufficiency: Yes.  "By the way, I'm really looking forward to finally owning a home, it's been a long process, but it'll be worth it to have a backyard like the one I'll have - remember when I got pre-approved for $400,000 from Wells Fargo?"
   use:         No. The response contains a mix of information from different contexts. While it correctly states that you were pre-approved for $350,000 this time, it incorrectly includes the $400,000 figure from your previous trip to Paris, which is not related to


[852ce960 / recent_2_sessions] pass=False location=retrieve
   reference facts: ['pre-approved for $350,000 from Wells Fargo', 'user was recently promoted to senior software engineer', 'aware that the series has historical inspirations from Renaissance Italy', 'previous 5-day trip to Paris last month was amazing', 'recently got car serviced on the 15th of last month', 'pre-approved for $400,000 from Wells Fargo']
   sufficiency: No.  The latest fact provided by the user regarding pre-approval from Wells Fargo is: "[2023/11/30 (Thu) 00:36] pre-approved for $400,000 from Wells Fargo". However, this information is not contained in the context given to the assistant.
   use:         No. The provided context does not contain any information about a mortgage pre-approval amount from Wells Fargo or any related financial details. The response correctly states that the assistant doesn't have access to specific user financial informat


[8550ddae / bm25_top3] pass=True location=None
   reference facts: ['looking for new vegan recipe ideas', 'tried a lavender gin fizz recipe last weekend', 'interested in the Lavender Dream cocktail', 'wants to create a signature cocktail for the event', 'plans to make simple syrup this weekend', "Completed 'The Last of Us Part II' on PlayStation 5 after playing for 10 hours last weekend"]
   sufficiency: Yes.  "Their" latest fact states: "[2023/05/22 (Mon) 08:41] tried a lavender gin fizz recipe last weekend."
   use:         Yes. The response accurately reflects the information provided in the context about trying a lavender gin fizz recipe last weekend and improvising with honey and water due to not having simple syrup.


[8550ddae / recent_2_sessions] pass=False location=retrieve
   reference facts: ['looking for new vegan recipe ideas', 'tried a lavender gin fizz recipe last weekend', 'interested in the Lavender Dream cocktail', 'wants to create a signature cocktail for the event', 'plans to make simple syrup this weekend', "Completed 'The Last of Us Part II' on PlayStation 5 after playing for 10 hours last weekend"]
   sufficiency: No.  The context does not contain any information about a cocktail recipe tried last weekend. The latest fact provided by the user is their interest in creating a signature cocktail for an event and planning to make simple syrup this weekend, but it 
   use:         No. The response does not answer the question using the information in the context because the context provided does not contain any details about cocktails or drinks that were tried last weekend.


## 7. Gold labels (only now): was memdiag right?
- `gold_correct`: Qwen grades the response against LongMemEval's answer (LongMemEval-style prompt).
- `gold_location`: from LongMemEval's evidence session ids, which need no model: if the evidence session
  wasn't in the context, a wrong answer is a **retrieve** failure; if it was, a **use** failure.

In [7]:
def gold_correct(item, response):
    extra = ("If the response contains some previous information along with an updated answer, it is correct "
             "as long as the updated answer is the required answer. "
             if item["question_type"] == "knowledge-update" else "")
    return yes(chat("gold_judge",
        "I will give you a question, a correct answer, and a response from a model. Answer yes if the response "
        f"contains the correct answer, otherwise answer no. {extra}\n\nQuestion: {item['question']}\n"
        f"Correct answer: {item['answer']}\nModel response: {response}\n\nIs the response correct? "
        "Answer yes or no only."))

rows = []
for c in calls:
    item = items[c["qid"]]
    ok = gold_correct(item, c["response"])
    ev = any(x["session"] in item["answer_session_ids"] for x in c["context"])
    g_loc = None if ok else ("use" if ev else "retrieve")
    m = c["memdiag"]
    rows.append({"qid": c["qid"], "config": c["config"], "gold_correct": ok, "memdiag_pass": m["pass"],
                 "detect_agrees": ok == m["pass"], "gold_location": g_loc, "memdiag_location": m["location"],
                 "location_agrees": None if ok else g_loc == m["location"]})

hdr = list(rows[0])
print(" | ".join(hdr))
for r in rows:
    print(" | ".join(str(r[h]) for h in hdr))

qid | config | gold_correct | memdiag_pass | detect_agrees | gold_location | memdiag_location | location_agrees
852ce960 | bm25_top3 | True | False | False | None | use | None
852ce960 | recent_2_sessions | False | False | True | retrieve | retrieve | True
8550ddae | bm25_top3 | True | True | True | None | None | None
8550ddae | recent_2_sessions | False | False | True | retrieve | retrieve | True


## 8. Summary: scores with denominators, and what Qwen was used for

In [8]:
n = len(rows)
wrong = [r for r in rows if not r["gold_correct"]]
print(f"detection agreement : {sum(r['detect_agrees'] for r in rows)}/{n}")
print(f"location agreement  : {sum(bool(r['location_agrees']) for r in wrong)}/{len(wrong)} (gold-wrong responses only)")
for cfg in CONFIGS:
    rs = [r for r in rows if r["config"] == cfg]
    print(f"{cfg:18s}: gold accuracy {sum(r['gold_correct'] for r in rs)}/{len(rs)}, "
          f"memdiag pass {sum(r['memdiag_pass'] for r in rs)}/{len(rs)}")

print("\nQwen calls by role (this run):")
by = defaultdict(lambda: [0, 0.0])
for e in LOG:
    by[e["role"]][0] += 1
    by[e["role"]][1] += e["secs"]
for role, (k, s) in by.items():
    print(f"  {role:18s} {k:3d} calls  {s:6.0f}s")
print("(the extractor shows 0 calls when the ledger came from cache)")

detection agreement : 3/4
location agreement  : 2/2 (gold-wrong responses only)
bm25_top3         : gold accuracy 2/2, memdiag pass 1/2
recent_2_sessions : gold accuracy 0/2, memdiag pass 0/2

Qwen calls by role (this run):
  smoke                1 calls      13s
  answerer             4 calls     223s
  judge_sufficiency    4 calls     269s
  judge_use            4 calls     185s
  gold_judge           4 calls      27s
(the extractor shows 0 calls when the ledger came from cache)


### Inspect any call
Change the role to see the exact prompt tail and output for that role.

In [9]:
for e in [e for e in LOG if e["role"] == "judge_sufficiency"][:2]:
    print("PROMPT ...", e["prompt"], "\nOUTPUT:", e["output"], "\n" + "-" * 60)

PROMPT ... ascinated by the art and history there. I actually just got back from a trip to Paris last month, and it was amazing!

Question: What was the amount I was pre-approved for when I got my mortgage from Wells Fargo?

Does the context contain the information needed to answer the question, consistent with the user's LATEST stated facts? Answer yes or no, then quote the supporting line from the context. 
OUTPUT: Yes.

"By the way, I'm really looking forward to finally owning a home, it's been a long process, but it'll be worth it to have a backyard like the one I'll have - remember when I got pre-approved for $400,000 from Wells Fargo?" 
------------------------------------------------------------
PROMPT ... 12/14 (Thu) 21:43] How do you find my questions?
- [2023/12/14 (Thu) 21:43] Do these issues have a common root cause?

Question: What was the amount I was pre-approved for when I got my mortgage from Wells Fargo?

Does the context contain the information needed to answer the q